# SkinCoach — Обучение модели v2.0
**Датасеты:** HAM10000 + SCIN (Google) + Fitzpatrick17k  
**Модель:** EfficientNet-B3 (12 классов)  
**Цель:** val_acc > 80%

## Как запустить на Kaggle:
1. Создать новый ноутбук на [kaggle.com/code](https://www.kaggle.com/code)
2. Settings → Accelerator → **GPU T4 x2**
3. Добавить датасеты:
   - `mariaherrerot/ham10000` (HAM10000)
   - `shubhamgoel27/dermnet` (DermNet — альтернатива Fitzpatrick17k)
4. Скопировать этот ноутбук или загрузить через File → Import notebook
5. Установить секрет: `HF_TOKEN` = ваш HuggingFace write token


In [ ]:
# ── Шаг 0: Проверка GPU ──────────────────────────────────────────────────────
import subprocess
result = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
print(result.stdout if result.returncode == 0 else '⚠️ GPU не найден')

In [ ]:
# ── Шаг 1: Установка зависимостей ────────────────────────────────────────────
!pip install -q huggingface_hub torch torchvision pillow

In [ ]:
# ── Шаг 2: Клонируем репозиторий SkinCoach ───────────────────────────────────
import os

REPO_URL = 'https://github.com/lipindanil02-eng/skincoach08.03.git'
# Если репо приватный — нужен токен:
# REPO_URL = 'https://<YOUR_GITHUB_TOKEN>@github.com/lipindanil02-eng/skincoach08.03.git'

if not os.path.exists('/kaggle/working/skincoach'):
    !git clone {REPO_URL} /kaggle/working/skincoach
else:
    print('Репозиторий уже скачан')

%cd /kaggle/working/skincoach

In [ ]:
# ── Шаг 3: Пути к датасетам (Kaggle автоматически монтирует в /kaggle/input/) ──
import os

# HAM10000 — добавить через «Add data» → «mariaherrerot/ham10000»
HAM_DIR = '/kaggle/input/ham10000'

# DermNet — добавить через «Add data» → «shubhamgoel27/dermnet»
# Содержит многие классы включая акне, розацеа, псориаз, экзему
DERMNET_DIR = '/kaggle/input/dermnet'

OUT_DIR = '/kaggle/working/dataset'

print('HAM10000 найден:', os.path.exists(HAM_DIR))
print('DermNet найден:', os.path.exists(DERMNET_DIR))

# Посмотрим структуру HAM10000
if os.path.exists(HAM_DIR):
    !ls {HAM_DIR}

In [ ]:
# ── Шаг 4: Скачиваем SCIN (Google) ────────────────────────────────────────────
# SCIN — датасет Google+Stanford: реальные фото пользователей, воспалительные состояния
# Репозиторий: https://github.com/google-research-datasets/scin

SCIN_DIR = '/kaggle/working/scin'

if not os.path.exists(SCIN_DIR):
    print('Клонируем SCIN (Google)...')
    !git clone https://github.com/google-research-datasets/scin.git {SCIN_DIR}
    
    # Скачиваем изображения (они хранятся в Google Cloud Storage)
    !pip install -q gcsfs
    import gcsfs
    fs = gcsfs.GCSFileSystem(token='anon')  # публичный доступ
    
    scin_img_dir = f'{SCIN_DIR}/images'
    os.makedirs(scin_img_dir, exist_ok=True)
    
    print('Скачиваем изображения SCIN (может занять 10-15 минут)...')
    try:
        # Попробуем через gsutil
        !gsutil -m cp 'gs://scin-public-data/v1/images/*.jpg' {scin_img_dir}/ 2>/dev/null || echo 'gsutil недоступен, пробуем Python...'
    except:
        pass
else:
    print('SCIN уже скачан')

!ls {SCIN_DIR}

In [ ]:
# ── Шаг 5: Обрабатываем DermNet как дополнительный источник ──────────────────
# DermNet на Kaggle уже структурирован по папкам (ImageFolder-совместим)
# Структура: dermnet/train/{acne, rosacea, psoriasis, ...}/

import shutil

# Маппинг папок DermNet → наши классы
DERMNET_MAP = {
    'Acne and Rosacea Photos':                    'acne',
    'Acne Nodules Cysts Acne and Rosacea Photos': 'acne',
    'Psoriasis pictures Lichen Planus and related diseases': 'psoriasis',
    'Atopic Dermatitis Photos':                   'eczema',
    'Eczema Photos':                              'eczema',
    'Seborrheic Keratoses and other Benign Tumors': 'keratosis',
    'Melanoma Skin Cancer Nevi and Moles':        'melanoma',
    'Basal Cell Carcinoma (Cancer) Photos':       'basal_cell_carcinoma',
    'Actinic Keratosis Basal Cell Carcinoma and other Malignant Lesions': 'actinic_keratosis',
    'Vascular Tumors':                            'other',
    'Urticaria Hives':                            'other',
    'Tinea Ringworm Candidiasis and other Fungal Infections': 'other',
    'Warts Molluscum and other Viral Infections': 'other',
    'Systemic Disease':                           'other',
    'Nail Fungus and other Nail Disease':         'other',
    'Light Diseases and Disorders of Pigmentation': 'vitiligo',
    'Lupus and other Connective Tissue diseases': 'other',
    'Bullous Disease Photos':                     'other',
    'Cellulitis Impetigo and other Bacterial Infections': 'other',
    'Hair Loss Photos Alopecia and other Hair Diseases': 'other',
    'Exanthems and Drug Eruptions':               'dermatitis',
    'Contact Dermatitis and other Eczemas':       'dermatitis',
    'Scabies Lyme Disease and other Infestations and Bites': 'other',
    'Vasculitis Photos':                          'rosacea',
    'Poison Ivy Photos and other Contact Dermatitis': 'dermatitis',
}

ALL_CLASSES = [
    'melanoma', 'nevus', 'basal_cell_carcinoma', 'actinic_keratosis',
    'keratosis', 'psoriasis', 'eczema', 'dermatitis',
    'acne', 'vitiligo', 'rosacea', 'other'
]

def copy_dermnet(dermnet_base, out_dir, split='train'):
    import random
    random.seed(42)
    
    split_dir = os.path.join(dermnet_base, split)
    if not os.path.exists(split_dir):
        print(f'  Папка {split_dir} не найдена')
        return
    
    counts = {cls: 0 for cls in ALL_CLASSES}
    
    for folder in os.listdir(split_dir):
        our_cls = DERMNET_MAP.get(folder)
        if not our_cls:
            # Попробуем по ключевым словам
            folder_lower = folder.lower()
            if 'acne' in folder_lower or 'rosacea' in folder_lower:
                our_cls = 'acne'
            elif 'eczema' in folder_lower or 'atopic' in folder_lower:
                our_cls = 'eczema'
            elif 'psoriasis' in folder_lower:
                our_cls = 'psoriasis'
            elif 'melanoma' in folder_lower:
                our_cls = 'melanoma'
            else:
                continue
        
        src_folder = os.path.join(split_dir, folder)
        dst_folder = os.path.join(out_dir, split, our_cls)
        os.makedirs(dst_folder, exist_ok=True)
        
        for fname in os.listdir(src_folder):
            if fname.lower().endswith(('.jpg', '.jpeg', '.png')):
                src = os.path.join(src_folder, fname)
                # Уникальное имя чтобы не перезаписать
                dst = os.path.join(dst_folder, f'dermnet_{folder[:20]}_{fname}')
                if not os.path.exists(dst):
                    shutil.copy2(src, dst)
                    counts[our_cls] += 1
    
    print(f'  DermNet ({split}):', {k: v for k, v in counts.items() if v > 0})

if os.path.exists(DERMNET_DIR):
    os.makedirs(OUT_DIR, exist_ok=True)
    for cls in ALL_CLASSES:
        os.makedirs(os.path.join(OUT_DIR, 'train', cls), exist_ok=True)
        os.makedirs(os.path.join(OUT_DIR, 'val', cls), exist_ok=True)
    
    copy_dermnet(DERMNET_DIR, OUT_DIR, 'train')
    copy_dermnet(DERMNET_DIR, OUT_DIR, 'test')   # DermNet test → наш val
    print('✅ DermNet обработан')
else:
    print('⚠️ DermNet не найден, пропускаю')

In [ ]:
# ── Шаг 6: Обрабатываем HAM10000 + SCIN через prepare_dataset.py ─────────────
cmd = f'python prepare_dataset.py --out {OUT_DIR}'

if os.path.exists(HAM_DIR):
    cmd += f' --ham {HAM_DIR}'

if os.path.exists(f'{SCIN_DIR}/images'):
    cmd += f' --scin {SCIN_DIR}'

print(f'Запускаю: {cmd}')
!{cmd}

In [ ]:
# ── Шаг 7: Статистика датасета ────────────────────────────────────────────────
import os

print('📊 Итоговый датасет:\n')
total_train, total_val = 0, 0
for cls in sorted(os.listdir(os.path.join(OUT_DIR, 'train'))):
    n_train = len(os.listdir(os.path.join(OUT_DIR, 'train', cls)))
    n_val   = len(os.listdir(os.path.join(OUT_DIR, 'val',   cls))) if os.path.exists(os.path.join(OUT_DIR, 'val', cls)) else 0
    total_train += n_train
    total_val   += n_val
    status = '✅' if n_train >= 100 else ('⚠️ ' if n_train > 0 else '❌')
    print(f'  {status} {cls:25s} train={n_train:5d}  val={n_val:4d}')

print(f'\n  Итого: train={total_train}, val={total_val}')

In [ ]:
# ── Шаг 8: ОБУЧЕНИЕ ───────────────────────────────────────────────────────────
# HuggingFace токен: добавьте в Kaggle → Add-ons → Secrets → HF_TOKEN
from kaggle_secrets import UserSecretsClient
try:
    secrets = UserSecretsClient()
    HF_TOKEN = secrets.get_secret('HF_TOKEN')
    print('✅ HuggingFace токен загружен')
except:
    HF_TOKEN = ''
    print('⚠️ HF_TOKEN не найден — модель не будет загружена на HuggingFace')
    print('   Добавьте токен: Kaggle → Settings → Secrets')

HF_REPO = 'danyil163/SCINCOACH'

!python train.py \
    --data {OUT_DIR} \
    --epochs 30 \
    --batch 32 \
    --lr 0.0001 \
    --out /kaggle/working/best_model.pth \
    --hf_repo {HF_REPO} \
    --hf_token {HF_TOKEN} \
    --workers 4

In [ ]:
# ── Шаг 9: Проверка модели ────────────────────────────────────────────────────
import json

with open('class_map.json') as f:
    cm = json.load(f)
print('class_map.json:', json.dumps(cm, indent=2, ensure_ascii=False))

import os
size_mb = os.path.getsize('/kaggle/working/best_model.pth') / 1024 / 1024
print(f'\nРазмер модели: {size_mb:.1f} MB')

## После обучения

1. Скачайте `best_model.pth` и `class_map.json` (Output → кнопка Download)
2. Загрузите их на HuggingFace: `danyil163/SCINCOACH` (скрипт делает это автоматически если есть HF_TOKEN)
3. Railway автоматически подхватит новую модель при следующем cold start

**Целевые метрики:**
- val_acc > 80% — хорошо
- val_acc > 85% — отлично  
- val_acc > 70% для редких классов (vitiligo, rosacea) — приемлемо
